In [115]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline  # Imbalanced-learn pipeline; supports SMOTE inside cross-validation.
from xgboost import XGBClassifier


def split_data(X, y, test_size=0.2, random_state=42):
    """Split features and labels while preserving the class ratio in each split."""
    return train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )


def make_preprocessor(numeric_features):
    """Scale the requested numeric columns and leave the rest unchanged."""
    return ColumnTransformer(
        transformers=[('num', StandardScaler(), numeric_features)],
        remainder='passthrough'
    )


def make_model_and_grid(model_name, random_state=42):
    """Return the estimator and hyperparameter grid for a supported model."""
    model_name = model_name.lower()
    if model_name == 'lr':
        # LogisticRegression now prefers l1_ratio over the deprecated penalty parameter.
        # We use saga so the same grid can cover L2 (l1_ratio=0), L1 (l1_ratio=1), and
        # intermediate elastic-net values without triggering deprecation warnings.
        estimator = LogisticRegression(random_state=random_state, max_iter=5000, solver='saga')
        param_grid = {
            'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
            'classifier__l1_ratio': [0.0, 0.25, 0.5, 0.75, 1.0],
        }
    elif model_name == 'rf':
        estimator = RandomForestClassifier(random_state=random_state)
        param_grid = {
            'classifier__n_estimators': [500, 600, 700],
            'classifier__max_depth': [20, 18, 15],
            'classifier__min_samples_split': [3, 5, 7],
            'classifier__min_samples_leaf': [1, 2, 4],
        }
    elif model_name == 'xgb':
        estimator = XGBClassifier(random_state=random_state, eval_metric='logloss')
        param_grid = {
            'classifier__learning_rate': [0.01, 0.05, 0.001],
            'classifier__n_estimators': [100, 80, 60],
            'classifier__max_depth': [7, 4, 6],
            'classifier__subsample': [0.8, 0.7, 0.5],
            'classifier__colsample_bytree': [0.8, 0.9, 1.0],
        }
    elif model_name == 'svm':
        # Calibrate SVC scores so we can use predict_proba without the deprecated probability flag.
        estimator = CalibratedClassifierCV(estimator=SVC(), ensemble=False)
        param_grid = [
            {
                'classifier__estimator__kernel': ['linear'],
                'classifier__estimator__C': [0.1, 1, 10, 100],
            },
            {
                'classifier__estimator__kernel': ['rbf'],
                'classifier__estimator__C': [0.1, 1, 10, 100],
                'classifier__estimator__gamma': ['scale', 'auto', 0.01, 0.1],
            },
        ]
    else:
        raise ValueError(f'Unsupported model_name: {model_name}')
    return estimator, param_grid


def run_model_experiment(X_train, X_test, y_train, y_test, model_name, numeric_features, scoring='f1_macro', cv=5, verbose=1, n_jobs=-1):
    """Run the shared preprocessing, SMOTE, grid search, and evaluation flow."""
    estimator, param_grid = make_model_and_grid(model_name)
    preprocessor = make_preprocessor(numeric_features)
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', estimator),
    ])
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring=scoring,
        verbose=verbose,
        n_jobs=n_jobs,
    )
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro'),
        'recall': recall_score(y_test, y_pred, average='macro'),
        'f1': f1_score(y_test, y_pred, average='macro'),
    }

    # Use score/probability outputs for ROC AUC so we do not evaluate on hard labels.
    y_score = best_model.predict_proba(X_test)[:, 1]
    metrics['roc_auc'] = roc_auc_score(y_test, y_score)

    print('Best parameters found:', grid_search.best_params_)
    print(f"Best Macro F1 Score: {grid_search.best_score_:.4f}")

    classifier = best_model.named_steps['classifier']
    feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
    if hasattr(classifier, 'coef_'):
        print('Model Coefficients:')
        for feature, coef in zip(feature_names, classifier.coef_[0]):
            print(f"{feature}: {coef}")
    elif hasattr(classifier, 'feature_importances_'):
        print('Feature Importances:')
        for feature, importance in zip(feature_names, classifier.feature_importances_):
            print(f"{feature}: {importance}")

    print(f"Evaluation Metrics for {model_name.upper()} with shared preprocessing and macro F1 grid search:")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1-score: {metrics['f1']:.4f}")
    if 'roc_auc' in metrics:
        print(f"ROC AUC: {metrics['roc_auc']:.4f}")

    return best_model, y_pred, metrics

In [116]:
columns = ['V2AF13', 'V2AF15', 'PublicID'] # father's age and years of education
df_paternal = pd.read_csv('../Data/V2A.csv', usecols=columns)
df_outcomes = pd.read_csv('../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])
df_outcomes

,PublicID,MH_outcome
0,00004O,1
1,00007I,1
2,00008G,0
3,00015J,0
4,00016H,1
...,...,...
7736,17349I,1
7737,17350A,1
7738,17351V,0
7739,17352T,1


In [117]:
df = pd.merge(df_paternal, df_outcomes, on='PublicID', how='inner')

In [118]:
# paternal age feature
df['V2AF13'].value_counts()

V2AF13
30    535
31    493
32    492
28    464
29    459
33    414
27    412
25    381
26    376
24    368
34    351
23    330
35    284
22    257
36    230
21    221
37    199
20    170
19    149
38    139
39     96
40     95
18     94
41     81
42     57
17     46
43     37
45     33
D      32
44     30
46     26
47     22
16     17
49     15
48     13
50     11
52      7
15      5
59      2
55      2
70      2
13      2
54      1
58      1
56      1
64      1
57      1
69      1
Name: count, dtype: int64

In [119]:
df = df[df != 'R']
df = df[df != 'D']
df['V2AF13'] = pd.to_numeric(df['V2AF13'], errors='coerce')
df['V2AF15'] = pd.to_numeric(df['V2AF15'], errors='coerce')
df = df.dropna(subset=['V2AF13', 'V2AF15', 'MH_outcome'])
df

,PublicID,V2AF13,V2AF15,MH_outcome
0,00004O,26.0,5.0,1
1,00007I,39.0,3.0,1
2,00008G,32.0,3.0,0
3,00015J,33.0,6.0,0
4,00016H,29.0,3.0,1
...,...,...,...,...
7603,17349I,25.0,5.0,1
7604,17350A,38.0,3.0,1
7605,17351V,32.0,6.0,0
7606,17352T,31.0,6.0,1


In [120]:
X = df.drop(['MH_outcome', 'PublicID'], axis=1)
y = df['MH_outcome']
y

0       1
1       1
2       0
3       0
4       1
       ..
7603    1
7604    1
7605    0
7606    1
7607    1
Name: MH_outcome, Length: 7353, dtype: int64

In [121]:
X

,V2AF13,V2AF15
0,26.0,5.0
1,39.0,3.0
2,32.0,3.0
3,33.0,6.0
4,29.0,3.0
...,...,...
7603,25.0,5.0
7604,38.0,3.0
7605,32.0,6.0
7606,31.0,6.0


In [122]:
y.value_counts()

MH_outcome
0    4403
1    2950
Name: count, dtype: int64

In [123]:
X_train, X_test, y_train, y_test = split_data(X, y)

# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    ['V2AF13', 'V2AF15']
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.01, 'classifier__l1_ratio': 1.0}
Best Macro F1 Score: 0.5642
Model Coefficients:
num__V2AF13: -0.0195008683492419
num__V2AF15: -0.22013051387122295
Evaluation Metrics for LR with shared preprocessing and macro F1 grid search:
Accuracy: 0.5710
Precision: 0.5655
Recall: 0.5680
F1-score: 0.5639
ROC AUC: 0.5850


In [124]:
X_train, X_test, y_train, y_test = split_data(X, y)

# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    ['V2AF13', 'V2AF15']
)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 20, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 700}
Best Macro F1 Score: 0.5444
Feature Importances:
num__V2AF13: 0.6447233948319163
num__V2AF15: 0.35527660516808357
Evaluation Metrics for RF with shared preprocessing and macro F1 grid search:
Accuracy: 0.5731
Precision: 0.5620
Recall: 0.5635
F1-score: 0.5619
ROC AUC: 0.5673


In [114]:
X_train, X_test, y_train, y_test = split_data(X, y)

# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    ['V2AF13', 'V2AF15']
)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.001, 'classifier__max_depth': 4, 'classifier__n_estimators': 100, 'classifier__subsample': 0.5}
Best Macro F1 Score: 0.5625
Feature Importances:
num__V2AF13: 0.17235970497131348
num__V2AF15: 0.8276402950286865
Evaluation Metrics for XGB with shared preprocessing and macro F1 grid search:
Accuracy: 0.5697
Precision: 0.5652
Recall: 0.5677
F1-score: 0.5631
ROC AUC: 0.5746


In [125]:
X_train, X_test, y_train, y_test = split_data(X, y)

# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    ['V2AF13', 'V2AF15']
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 10, 'classifier__estimator__gamma': 0.1, 'classifier__estimator__kernel': 'rbf'}
Best Macro F1 Score: 0.5649
Evaluation Metrics for SVM with shared preprocessing and macro F1 grid search:
Accuracy: 0.5799
Precision: 0.5726
Recall: 0.5751
F1-score: 0.5717
ROC AUC: 0.5819
